In [1]:
# Setup
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJ      = Path.cwd().parent
data_dir  = PROJ / 'data'
fig_dir   = PROJ / 'figures'
fig_dir.mkdir(exist_ok=True)

EMBD_COLS = [f'A{i:02d}' for i in range(64)]
print(f'Data: {data_dir}')
print(f'Embedding dims: {len(EMBD_COLS)}')

Data: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data
Embedding dims: 64


> **Ch.11 — AlphaEarth × Metal Hotspot Synthesis** 
>
> **Scientific question**: Do NCBI reference genomes co-located with geographic metal-resistance
> hotspots (Ch.10b, 5° grid) occupy distinct regions of AlphaEarth embedding space? If genomes
> from hotspot grid cells cluster separately in the 64-dim learned representation, this suggests
> environmental metal exposure shapes genomic content detectable beyond explicit resistance genes.
>
> **Data**: `alphaearth_with_env.csv` — 83K NCBI reference genomes with 64-dim AlphaEarth
> embeddings (A00–A63), lat/lon, and environmental metadata. Spatial join to `hotspots_5grid.csv`
> at 5° grid resolution (same binning as Ch.10b).

In [2]:
# Load AlphaEarth and filter to environmental genomes with coordinates

ae = pd.read_csv(data_dir / 'alphaearth_with_env.csv')
print(f'Total AlphaEarth genomes: {len(ae)}')

# Restrict to genomes with lat/lon
ae = ae[ae['has_latlon'] == True].dropna(subset=['cleaned_lat', 'cleaned_lon'])
print(f'With lat/lon: {len(ae)}')

# Exclude host-associated genomes using isolation_source, env_broad_scale, and host
HUMAN_KEYWORDS = [
    'human', 'clinical', 'gut', 'feces', 'fecal', 'blood', 'sputum',
    'urine', 'skin', 'bodily', 'body', 'patient', 'hospital', 'stool', 'wound'
]

def is_environmental(row):
    iso  = str(row.get('isolation_source', '') or '').lower()
    env  = str(row.get('env_broad_scale',  '') or '').lower()
    host = str(row.get('host',             '') or '').lower()
    if any(k in iso  for k in HUMAN_KEYWORDS): return False
    if any(k in env  for k in HUMAN_KEYWORDS): return False
    if 'homo sapiens' in host or 'human' in host: return False
    return True

env_mask = ae.apply(is_environmental, axis=1)
ae_env = ae[env_mask].copy()
print(f'Environmental genomes: {len(ae_env)} ({env_mask.mean():.1%} of total with coords)')

# Verify embeddings complete
missing_embd = ae_env[EMBD_COLS].isnull().any(axis=1).sum()
print(f'Genomes missing any embedding dim: {missing_embd}')
ae_env = ae_env.dropna(subset=EMBD_COLS)

Total AlphaEarth genomes: 83287
With lat/lon: 83286


Environmental genomes: 40552 (48.7% of total with coords)
Genomes missing any embedding dim: 3581


In [3]:
# Spatial join: label genomes as hotspot vs non-hotspot using 5° grid

hotspots = pd.read_csv(data_dir / 'hotspots_5grid.csv')
print(f'Hotspot cells: {len(hotspots)}')
print(hotspots[['lat_bin', 'lon_bin', 'OR', 'q']].to_string(index=False))

# Same 5° binning used in Ch.10b
ae_env['lat_bin'] = (ae_env['cleaned_lat'] // 5) * 5
ae_env['lon_bin'] = (ae_env['cleaned_lon'] // 5) * 5

hotspot_cells = set(zip(hotspots['lat_bin'], hotspots['lon_bin']))
ae_env['is_hotspot'] = ae_env.apply(
    lambda r: (r['lat_bin'], r['lon_bin']) in hotspot_cells, axis=1
)

n_hs  = ae_env['is_hotspot'].sum()
n_nhs = (~ae_env['is_hotspot']).sum()
print(f'\nHotspot genomes:     {n_hs:>6,}')
print(f'Non-hotspot genomes: {n_nhs:>6,}')
print(f'Fraction hotspot:    {n_hs/(n_hs+n_nhs):.3f}')

# Per-cell counts
cell_counts = (
    ae_env[ae_env['is_hotspot']]
    .groupby(['lat_bin', 'lon_bin'])
    .size()
    .reset_index(name='n_genomes')
)
print(f'\nGenomes per hotspot cell:')
print(cell_counts.to_string(index=False))

Hotspot cells: 11
 lat_bin  lon_bin       OR            q
   -25.0    -70.0 9.831788 7.599228e-12
    40.0    -80.0 7.860702 2.752579e-11
    40.0    -90.0 6.323634 7.599228e-12
    30.0     85.0 6.269114 1.673747e-02
    25.0    105.0 5.894050 1.014822e-02
    25.0    115.0 4.429132 5.706611e-04
    30.0    120.0 3.773459 1.898187e-03
    25.0    -85.0 2.710944 1.759981e-03
    50.0     10.0 2.647750 3.901725e-03
    30.0   -120.0 2.570332 1.673747e-02
    35.0   -125.0 2.480344 1.671870e-05



Hotspot genomes:      4,496
Non-hotspot genomes: 32,475
Fraction hotspot:    0.122

Genomes per hotspot cell:
 lat_bin  lon_bin  n_genomes
   -25.0    -70.0        177
    25.0    -85.0        127
    25.0    105.0        342
    25.0    115.0        186
    30.0   -120.0        192
    30.0     85.0         10
    30.0    120.0        527
    35.0   -125.0       1129
    40.0    -90.0       1297
    40.0    -80.0        235
    50.0     10.0        274


In [4]:
# PCA on 64-dim AlphaEarth embeddings

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = ae_env[EMBD_COLS].values

# Standardize before PCA (embeddings may have different scales)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=20, random_state=42)
pcs = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print('Variance explained by first 10 PCs:')
for i, ev in enumerate(explained[:10]):
    print(f'  PC{i+1:02d}: {ev:.3f} ({explained[:i+1].sum():.3f} cumulative)')

# Add PCs back to dataframe
for k in range(20):
    ae_env[f'PC{k+1}'] = pcs[:, k]

print(f'\nPCA complete: {X.shape[0]} genomes × {X.shape[1]} dims → 20 PCs')

Variance explained by first 10 PCs:
  PC01: 0.160 (0.160 cumulative)
  PC02: 0.086 (0.246 cumulative)
  PC03: 0.073 (0.320 cumulative)
  PC04: 0.067 (0.386 cumulative)
  PC05: 0.053 (0.440 cumulative)
  PC06: 0.050 (0.490 cumulative)
  PC07: 0.037 (0.526 cumulative)
  PC08: 0.031 (0.558 cumulative)
  PC09: 0.031 (0.588 cumulative)
  PC10: 0.026 (0.614 cumulative)

PCA complete: 36971 genomes × 64 dims → 20 PCs


In [5]:
# PERMANOVA: test whether hotspot and non-hotspot genomes differ in PC space

import skbio
from skbio.stats.distance import permanova, DistanceMatrix
from scipy.spatial.distance import cdist

# Use first 10 PCs (captures ~majority of variance)
N_PCS_FOR_TEST = 10

# Subsample for PERMANOVA if too large (PERMANOVA scales as O(n²))
MAX_N = 5000
np.random.seed(42)
df_test = ae_env.copy()
if len(df_test) > MAX_N:
    # Stratified subsample: preserve hotspot:non-hotspot ratio
    n_hs_sub  = min(n_hs,  int(MAX_N * n_hs  / (n_hs + n_nhs)))
    n_nhs_sub = MAX_N - n_hs_sub
    hs_idx  = df_test[df_test['is_hotspot']].sample(n=n_hs_sub, random_state=42).index
    nhs_idx = df_test[~df_test['is_hotspot']].sample(n=n_nhs_sub, random_state=42).index
    df_test = df_test.loc[hs_idx.union(nhs_idx)]
    print(f'Subsampled to {len(df_test)} genomes for PERMANOVA '
          f'({df_test["is_hotspot"].sum()} hotspot, {(~df_test["is_hotspot"]).sum()} non-hotspot)')
else:
    print(f'Using all {len(df_test)} genomes for PERMANOVA')

pc_cols = [f'PC{k+1}' for k in range(N_PCS_FOR_TEST)]
X_test = df_test[pc_cols].values

# Euclidean distance matrix in PC space
dist_mat = cdist(X_test, X_test, metric='euclidean')
dm = DistanceMatrix(dist_mat, ids=[str(i) for i in range(len(df_test))])

grouping = df_test['is_hotspot'].map({True: 'hotspot', False: 'non_hotspot'}).values
result = permanova(dm, grouping, permutations=999)

print('\nPERMANOVA result (Euclidean distance in PC1–10 space):')
print(result)

f_stat   = result['test statistic']
p_val    = result['p-value']
print(f'\nF-statistic: {f_stat:.4f}')
print(f'p-value:     {p_val:.4f}')

Subsampled to 5000 genomes for PERMANOVA (608 hotspot, 4392 non-hotspot)



PERMANOVA result (Euclidean distance in PC1–10 space):
method name               PERMANOVA
test statistic name        pseudo-F
sample size                    5000
number of groups                  2
test statistic             80.68059
p-value                       0.001
number of permutations          999
Name: PERMANOVA results, dtype: object

F-statistic: 80.6806
p-value:     0.0010


In [6]:
# Per-PC univariate test: Welch t-test on each PC between hotspot and non-hotspot

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

pc_results = []
for k in range(20):
    pc_col = f'PC{k+1}'
    hs_vals  = ae_env.loc[ae_env['is_hotspot'],  pc_col].values
    nhs_vals = ae_env.loc[~ae_env['is_hotspot'], pc_col].values
    t, p = ttest_ind(hs_vals, nhs_vals, equal_var=False)
    pc_results.append({'PC': pc_col, 't_stat': t, 'p_raw': p,
                       'mean_hotspot': hs_vals.mean(),
                       'mean_nonhotspot': nhs_vals.mean(),
                       'diff': hs_vals.mean() - nhs_vals.mean()})

pc_df = pd.DataFrame(pc_results)
_, pc_df['q'], _, _ = multipletests(pc_df['p_raw'], method='fdr_bh')
pc_df['sig'] = pc_df['q'] < 0.05

print('Per-PC Welch t-test (hotspot vs non-hotspot):')
print(pc_df[['PC', 'mean_hotspot', 'mean_nonhotspot', 'diff', 't_stat', 'p_raw', 'q', 'sig']].to_string(index=False))

n_sig = pc_df['sig'].sum()
print(f'\nSignificant PCs (q<0.05): {n_sig} / {len(pc_df)}')

Per-PC Welch t-test (hotspot vs non-hotspot):
  PC  mean_hotspot  mean_nonhotspot      diff     t_stat         p_raw             q   sig
 PC1     -1.374772         0.190330 -1.565102 -44.209670  0.000000e+00  0.000000e+00  True
 PC2     -0.171782         0.023782 -0.195565  -4.705660  2.592708e-06  4.714015e-06  True
 PC3      0.159036        -0.022018  0.181054   5.768454  8.386180e-09  1.863596e-08  True
 PC4      0.034970        -0.004841  0.039811   1.460399  1.442271e-01  1.602523e-01 False
 PC5      0.382516        -0.052957  0.435473  23.058987 1.679615e-114 5.598718e-114  True
 PC6     -0.102216         0.014151 -0.116367  -3.435837  5.952574e-04  8.503677e-04  True
 PC7      1.496273        -0.207151  1.703424  63.131577  0.000000e+00  0.000000e+00  True
 PC8      0.056382        -0.007806  0.064188   3.217841  1.298076e-03  1.730768e-03  True
 PC9      0.175304        -0.024270  0.199573   7.694471  1.680489e-14  4.201222e-14  True
PC10     -0.051994         0.007198 -0.05919

In [7]:
# Visualization: PCA scatter + per-PC bar chart

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: PC1 vs PC2 scatter ---
ax = axes[0]
# Subsample non-hotspot for visibility (plot at most 5000)
nhs_plot = ae_env[~ae_env['is_hotspot']].sample(n=min(5000, n_nhs), random_state=0)
hs_plot  = ae_env[ae_env['is_hotspot']]

ax.scatter(nhs_plot['PC1'], nhs_plot['PC2'], color='#adb5bd', alpha=0.15, s=2,
           label=f'Non-hotspot (n={n_nhs:,}; showing 5k)', rasterized=True)
ax.scatter(hs_plot['PC1'], hs_plot['PC2'], color='#e63946', alpha=0.5, s=5,
           label=f'Hotspot (n={n_hs:,})', rasterized=True)

ax.set_xlabel(f'PC1 ({explained[0]:.1%} var)', fontsize=12)
ax.set_ylabel(f'PC2 ({explained[1]:.1%} var)', fontsize=12)
ax.set_title('AlphaEarth Embedding Space\nHotspot vs Non-Hotspot Genomes', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, markerscale=3)
ax.grid(True, alpha=0.3)

# --- Right: per-PC effect size (Welch t-statistic) with significance marking ---
ax2 = axes[1]
colors = ['#e63946' if sig else '#adb5bd' for sig in pc_df['sig']]
bars = ax2.barh(pc_df['PC'], pc_df['t_stat'].abs(), color=colors)
ax2.set_xlabel('|t-statistic| (Welch t-test: hotspot vs non-hotspot)', fontsize=11)
ax2.set_title('Per-PC Effect Size\n(BH-FDR q<0.05 in red)', fontsize=12, fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.5)
ax2.grid(True, axis='x', alpha=0.3)

from matplotlib.patches import Patch
ax2.legend(handles=[
    Patch(color='#e63946', label='q<0.05 (FDR)'),
    Patch(color='#adb5bd', label='q≥0.05')
], fontsize=9)

plt.tight_layout()
out = fig_dir / 'nb11c_alphaearth_pca_hotspot.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'Saved: {out}')
plt.close()

# --- PC1 distribution comparison ---
fig2, ax3 = plt.subplots(figsize=(10, 5))
ax3.hist(ae_env.loc[~ae_env['is_hotspot'], 'PC1'], bins=80, density=True,
         alpha=0.5, color='#adb5bd', label=f'Non-hotspot (n={n_nhs:,})')
ax3.hist(ae_env.loc[ae_env['is_hotspot'], 'PC1'], bins=80, density=True,
         alpha=0.7, color='#e63946', label=f'Hotspot (n={n_hs:,})')
ax3.set_xlabel(f'PC1 ({explained[0]:.1%} variance explained)', fontsize=12)
ax3.set_ylabel('Density', fontsize=12)
ax3.set_title(f'PC1 Distribution: Hotspot vs Non-Hotspot\n'
              f'Welch t={pc_df.loc[pc_df["PC"]=="PC1", "t_stat"].values[0]:.2f}, '
              f'q={pc_df.loc[pc_df["PC"]=="PC1", "q"].values[0]:.3g}',
              fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
out2 = fig_dir / 'nb11c_alphaearth_pc1_distribution.png'
fig2.savefig(out2, dpi=300, bbox_inches='tight')
print(f'Saved: {out2}')
plt.close()

Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_alphaearth_pca_hotspot.png


Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_alphaearth_pc1_distribution.png


In [8]:
# Save results

out_df = ae_env[['genome_id', 'cleaned_lat', 'cleaned_lon', 'lat_bin', 'lon_bin',
                  'is_hotspot', 'domain', 'phylum', 'genus', 'species',
                  'isolation_source', 'env_broad_scale'] +
                 [f'PC{k+1}' for k in range(20)]].copy()
out_path = data_dir / 'alphaearth_hotspot_comparison.csv'
out_df.to_csv(out_path, index=False)
print(f'Saved: {out_path} ({len(out_df)} rows)')

# Summary statistics
print('\n=== PERMANOVA result ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value:     {p_val:.4f}')

print('\n=== Significant PCs (q<0.05) ===')
print(pc_df.loc[pc_df['sig'], ['PC', 'diff', 't_stat', 'q']].to_string(index=False))

print('\n=== Variance explained by PC1-5 ===')
for k in range(5):
    print(f'  PC{k+1}: {explained[k]:.3f} ({explained[:k+1].sum():.3f} cumulative)')

Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/alphaearth_hotspot_comparison.csv (36971 rows)

=== PERMANOVA result ===
F-statistic: 80.6806
p-value:     0.0010

=== Significant PCs (q<0.05) ===
  PC      diff     t_stat             q
 PC1 -1.565102 -44.209670  0.000000e+00
 PC2 -0.195565  -4.705660  4.714015e-06
 PC3  0.181054   5.768454  1.863596e-08
 PC5  0.435473  23.058987 5.598718e-114
 PC6 -0.116367  -3.435837  8.503677e-04
 PC7  1.703424  63.131577  0.000000e+00
 PC8  0.064188   3.217841  1.730768e-03
 PC9  0.199573   7.694471  4.201222e-14
PC10 -0.059193  -2.670207  9.503212e-03
PC12  0.453970  25.363146 1.754259e-134
PC14  0.190266  10.162683  1.335567e-23
PC15 -0.090105  -4.936551  1.635375e-06
PC16 -0.082755  -4.209600  4.333399e-05
PC17 -0.445325 -29.870247 2.230818e-182
PC19 -0.402164 -24.417177 1.443831e-124
PC20  0.054592   3.590316  5.124077e-04

=== Variance explained by PC1-5 ===
  PC1: 0.160 (0.160 cumulative)
  PC2: 0.08

---
## Extension 1: Do AlphaEarth PCs encode metal resistance?

**Question**: If the PC dimensions most shifted in hotspot genomes actually capture metal resistance biology, they should correlate with genus-level metal resistance traits *globally* — not just in hotspot cells. Join AlphaEarth genera to `genus_trait_table.csv` (2,851 GTDB genera, each with `mean_n_metal_types`) and test Spearman correlation of mean PC values per genus vs metal resistance load.

In [9]:
# Extension 1: PC vs genus-level metal resistance

from scipy.stats import spearmanr

# Normalise AE genus labels to lowercase without GTDB prefix
ae_env['genus_lower'] = (ae_env['genus']
    .str.replace(r'^g__', '', regex=True)
    .str.lower()
    .str.strip())

traits = pd.read_csv(data_dir / 'genus_trait_table.csv')

# Significant PCs (strong hotspot signal): 1, 5, 7, 12, 17, 19
SIG_PCS = ['PC1', 'PC5', 'PC7', 'PC12', 'PC17', 'PC19']

# Per-genus mean PC values — require ≥ 5 genomes for stable mean
genus_pcs = (ae_env
    .groupby('genus_lower')[SIG_PCS]
    .agg(['mean', 'count']))
genus_pcs.columns = ['_'.join(c) for c in genus_pcs.columns]
genus_pcs = genus_pcs[genus_pcs['PC1_count'] >= 5].reset_index()

# Join to trait table
genus_joined = genus_pcs.merge(
    traits[['genus_lower', 'mean_n_metal_types', 'mean_n_metal_clusters']],
    on='genus_lower', how='inner')
print(f'Genera with ≥5 AE genomes and trait data: {len(genus_joined)}')
print(f'Metal type range: {genus_joined["mean_n_metal_types"].min():.2f} – '
      f'{genus_joined["mean_n_metal_types"].max():.2f}')

# Spearman correlations for each significant PC vs metal resistance
print('\nSpearman ρ (per-genus mean PC vs mean_n_metal_types):')
corr_rows = []
for pc in SIG_PCS:
    col = f'{pc}_mean'
    r, p = spearmanr(genus_joined[col], genus_joined['mean_n_metal_types'],
                     nan_policy='omit')
    corr_rows.append({'PC': pc, 'rho': r, 'p': p, 'sig': p < 0.05})
    star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    print(f'  {pc}: ρ={r:+.3f}  p={p:.3g} {star}')

corr_df = pd.DataFrame(corr_rows)

# Top and bottom quartile genera by metal resistance — do their PCs differ?
q25 = genus_joined['mean_n_metal_types'].quantile(0.25)
q75 = genus_joined['mean_n_metal_types'].quantile(0.75)
hi_metal = genus_joined[genus_joined['mean_n_metal_types'] >= q75]
lo_metal = genus_joined[genus_joined['mean_n_metal_types'] <= q25]
print(f'\nTop quartile (metal ≥ {q75:.2f}): {len(hi_metal)} genera')
print(f'Bot quartile (metal ≤ {q25:.2f}): {len(lo_metal)} genera')

for pc in ['PC1', 'PC7']:
    col = f'{pc}_mean'
    from scipy.stats import ttest_ind as _t
    t, p = _t(hi_metal[col].dropna(), lo_metal[col].dropna(), equal_var=False)
    print(f'  {pc} hi vs lo: t={t:.2f}  p={p:.3g}')

Genera with ≥5 AE genomes and trait data: 394
Metal type range: 1.00 – 6.38

Spearman ρ (per-genus mean PC vs mean_n_metal_types):
  PC1: ρ=+0.038  p=0.458 
  PC5: ρ=+0.011  p=0.827 
  PC7: ρ=-0.065  p=0.204 
  PC12: ρ=+0.029  p=0.569 
  PC17: ρ=+0.041  p=0.421 
  PC19: ρ=+0.032  p=0.538 

Top quartile (metal ≥ 3.31): 95 genera
Bot quartile (metal ≤ 1.67): 100 genera
  PC1 hi vs lo: t=-0.21  p=0.837
  PC7 hi vs lo: t=-1.59  p=0.112


In [10]:
# Extension 1 — Figure: genus mean PC1 & PC7 vs metal resistance

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pc, label in zip(axes, ['PC1', 'PC7'],
                          [f'PC1 ({explained[0]:.1%} var)', f'PC7 ({explained[6]:.1%} var)']):
    col = f'{pc}_mean'
    r = corr_df.loc[corr_df['PC'] == pc, 'rho'].values[0]
    p = corr_df.loc[corr_df['PC'] == pc, 'p'].values[0]

    sc = ax.scatter(genus_joined['mean_n_metal_types'], genus_joined[col],
                    alpha=0.4, s=genus_joined['PC1_count'] * 0.5,
                    c=genus_joined['mean_n_metal_types'], cmap='plasma',
                    edgecolors='none', rasterized=True)

    # Trend line
    m, b = np.polyfit(genus_joined['mean_n_metal_types'].fillna(0),
                      genus_joined[col].fillna(0), 1)
    xs = np.linspace(genus_joined['mean_n_metal_types'].min(),
                     genus_joined['mean_n_metal_types'].max(), 100)
    ax.plot(xs, m * xs + b, color='black', linewidth=1.5, linestyle='--')

    star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    ax.set_xlabel('Genus mean metal resistance types', fontsize=12)
    ax.set_ylabel(f'Genus mean {label}', fontsize=12)
    ax.set_title(f'{label} vs Metal Resistance\nSpearman ρ={r:+.3f}  {star}',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.colorbar(sc, ax=ax, label='Metal resistance types')

plt.tight_layout()
out = fig_dir / 'nb11c_ext1_pc_vs_metal.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'Saved: {out}')
plt.close()

Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_ext1_pc_vs_metal.png


---
## Extension 2: Taxonomic composition control

**Confound**: Hotspot grid cells fall in specific geographic regions (South America, North America, East Asia, Europe). If those regions happen to have different microbial community compositions (e.g., more Proteobacteria), the PC separation may reflect taxonomy rather than metal exposure. Two tests: (1) phylum composition chi-square to quantify the confound; (2) within-Pseudomonadota PERMANOVA to see if the signal survives within the dominant phylum.

In [11]:
# Extension 2a: Phylum composition — hotspot vs non-hotspot

from scipy.stats import chi2_contingency

ae_env['phylum_clean'] = ae_env['phylum'].str.replace(r'^p__', '', regex=True)

# Top phyla (≥1% of total)
top_phyla = (ae_env['phylum_clean'].value_counts(normalize=True)
             .loc[lambda s: s >= 0.01].index.tolist())

hs_phylum  = ae_env[ae_env['is_hotspot']]['phylum_clean'].value_counts()
nhs_phylum = ae_env[~ae_env['is_hotspot']]['phylum_clean'].value_counts()

comp = pd.DataFrame({
    'hotspot': hs_phylum,
    'non_hotspot': nhs_phylum
}).fillna(0).loc[top_phyla].sort_index()

comp['pct_hotspot']     = comp['hotspot']     / comp['hotspot'].sum() * 100
comp['pct_non_hotspot'] = comp['non_hotspot'] / comp['non_hotspot'].sum() * 100
comp['pct_diff']        = comp['pct_hotspot'] - comp['pct_non_hotspot']

print('Phylum composition — hotspot vs non-hotspot (%):')
print(comp[['pct_hotspot', 'pct_non_hotspot', 'pct_diff']].round(1).to_string())

# Chi-square test on raw counts
chi2, p_comp, dof, _ = chi2_contingency(comp[['hotspot', 'non_hotspot']].values)
print(f'\nChi-square test: χ²={chi2:.1f}, df={dof}, p={p_comp:.3g}')
print('(significant ⟹ phylum composition IS confounded with hotspot status)')

# Figure: stacked bar comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top_phyla))
w = 0.35
ax.bar(x - w/2, comp['pct_hotspot'],     w, label='Hotspot',     color='#e63946', alpha=0.8)
ax.bar(x + w/2, comp['pct_non_hotspot'], w, label='Non-hotspot', color='#adb5bd', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([p.replace('Pseudomonadota', 'Proteobacteria\n(Pseudomonadota)')
                    for p in top_phyla], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('% of group', fontsize=12)
ax.set_title(f'Phylum Composition: Hotspot vs Non-Hotspot\nχ²={chi2:.1f} p={p_comp:.3g}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
out = fig_dir / 'nb11c_ext2_phylum_composition.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'\nSaved: {out}')
plt.close()

Phylum composition — hotspot vs non-hotspot (%):
                   pct_hotspot  pct_non_hotspot  pct_diff
phylum_clean                                             
Acidobacteriota            3.2              1.8       1.3
Actinomycetota            10.3              9.2       1.0
Bacillota                 17.3             12.8       4.5
Bacillota_A                3.9              2.2       1.7
Bacteroidota               7.1              8.9      -1.8
Campylobacterota           0.8              1.8      -1.0
Chloroflexota              3.3              2.1       1.2
Cyanobacteriota            3.2              2.1       1.0
Desulfobacterota           1.0              1.4      -0.4
Halobacteriota             1.5              1.1       0.4
Patescibacteria            2.4              7.1      -4.7
Planctomycetota            1.2              1.5      -0.4
Pseudomonadota            38.7             43.9      -5.1
Thermoplasmatota           1.8              1.2       0.6
Thermoproteota         


Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_ext2_phylum_composition.png


In [12]:
# Extension 2b: Within-Pseudomonadota PERMANOVA (controls for taxonomy)

proto = ae_env[ae_env['phylum'] == 'p__Pseudomonadota'].copy()
n_hs_proto  = proto['is_hotspot'].sum()
n_nhs_proto = (~proto['is_hotspot']).sum()
print(f'Pseudomonadota genomes: {len(proto)} ({n_hs_proto} hotspot, {n_nhs_proto} non-hotspot)')

# Stratified subsample (same MAX_N = 5000)
np.random.seed(42)
n_hs_sub  = min(n_hs_proto,  int(MAX_N * n_hs_proto  / (n_hs_proto + n_nhs_proto)))
n_nhs_sub = min(n_nhs_proto, MAX_N - n_hs_sub)
hs_idx  = proto[proto['is_hotspot']].sample(n=n_hs_sub, random_state=42).index
nhs_idx = proto[~proto['is_hotspot']].sample(n=n_nhs_sub, random_state=42).index
proto_sub = proto.loc[hs_idx.union(nhs_idx)]
print(f'Subsampled to {len(proto_sub)} ({proto_sub["is_hotspot"].sum()} hotspot)')

X_proto = proto_sub[pc_cols].values
dm_proto = DistanceMatrix(cdist(X_proto, X_proto, 'euclidean'),
                           ids=[str(i) for i in range(len(proto_sub))])
grp_proto = proto_sub['is_hotspot'].map({True: 'hotspot', False: 'non_hotspot'}).values
res_proto = permanova(dm_proto, grp_proto, permutations=999)

f_proto = res_proto['test statistic']
p_proto = res_proto['p-value']
print(f'\nWithin-Pseudomonadota PERMANOVA:')
print(f'  pseudo-F: {f_proto:.4f}')
print(f'  p-value:  {p_proto:.4f}')

# Repeat for Actinomycetota as independent check
actino = ae_env[ae_env['phylum'] == 'p__Actinomycetota'].copy()
n_hs_ac  = actino['is_hotspot'].sum()
n_nhs_ac = (~actino['is_hotspot']).sum()
print(f'\nActinomycetota: {len(actino)} ({n_hs_ac} hotspot, {n_nhs_ac} non-hotspot)')

if n_hs_ac >= 20:
    n_hs_sub2  = min(n_hs_ac,  int(MAX_N * n_hs_ac  / (n_hs_ac + n_nhs_ac)))
    n_nhs_sub2 = min(n_nhs_ac, MAX_N - n_hs_sub2)
    hs_idx2  = actino[actino['is_hotspot']].sample(n=n_hs_sub2, random_state=42).index
    nhs_idx2 = actino[~actino['is_hotspot']].sample(n=n_nhs_sub2, random_state=42).index
    actino_sub = actino.loc[hs_idx2.union(nhs_idx2)]
    X_ac = actino_sub[pc_cols].values
    dm_ac = DistanceMatrix(cdist(X_ac, X_ac, 'euclidean'),
                            ids=[str(i) for i in range(len(actino_sub))])
    grp_ac = actino_sub['is_hotspot'].map({True: 'hotspot', False: 'non_hotspot'}).values
    res_ac = permanova(dm_ac, grp_ac, permutations=999)
    print(f'Within-Actinomycetota PERMANOVA:')
    print(f'  pseudo-F: {res_ac["test statistic"]:.4f}')
    print(f'  p-value:  {res_ac["p-value"]:.4f}')
else:
    print('Too few hotspot genomes for PERMANOVA')
    res_ac = None

print('\n--- Interpretation ---')
print(f'Overall PERMANOVA F={f_stat:.2f}; Within-Pseudomonadota F={f_proto:.2f}')
print('If within-phylum F remains large, taxonomy does NOT explain the hotspot signal.')

Pseudomonadota genomes: 14172 (1497 hotspot, 12675 non-hotspot)
Subsampled to 5000 (528 hotspot)



Within-Pseudomonadota PERMANOVA:
  pseudo-F: 96.1393
  p-value:  0.0010

Actinomycetota: 3065 (397 hotspot, 2668 non-hotspot)


Within-Actinomycetota PERMANOVA:
  pseudo-F: 95.1326
  p-value:  0.0010

--- Interpretation ---
Overall PERMANOVA F=80.68; Within-Pseudomonadota F=96.14
If within-phylum F remains large, taxonomy does NOT explain the hotspot signal.


---
## Extension 3: Dose-response — hotspot OR vs PC displacement

**Question**: If the PC signal tracks metal exposure rather than some other geographic feature, cells with *stronger* metal-resistance enrichment (higher OR from Ch.10b Fisher's test) should show *larger* PC displacement. This within-hotspot dose-response is harder to explain by taxonomy or biogeography alone.

In [13]:
# Extension 3: Hotspot OR vs PC displacement (dose-response)

from scipy.stats import pearsonr, spearmanr

# Global non-hotspot baseline for each significant PC
baseline = {pc: ae_env.loc[~ae_env['is_hotspot'], pc].mean() for pc in SIG_PCS}

# Per hotspot cell: mean PC values
cell_pc = (ae_env[ae_env['is_hotspot']]
           .groupby(['lat_bin', 'lon_bin'])[SIG_PCS]
           .mean()
           .reset_index())

# Compute displacement vs baseline
for pc in SIG_PCS:
    cell_pc[f'{pc}_displacement'] = cell_pc[pc] - baseline[pc]

# Merge with hotspot OR
hotspots_reload = pd.read_csv(data_dir / 'hotspots_5grid.csv')
cell_pc = cell_pc.merge(hotspots_reload[['lat_bin', 'lon_bin', 'OR', 'q', 'n_total']],
                         on=['lat_bin', 'lon_bin'])
cell_pc['log2_OR'] = np.log2(cell_pc['OR'])

print('Per-cell displacement vs OR:')
print(cell_pc[['lat_bin', 'lon_bin', 'OR', 'n_total',
               'PC1_displacement', 'PC7_displacement']].round(3).to_string(index=False))

# Spearman / Pearson for each PC
print('\nCorrelation: |PC displacement| vs log2(OR) across 11 hotspot cells:')
dose_rows = []
for pc in SIG_PCS:
    col = f'{pc}_displacement'
    r_s, p_s = spearmanr(cell_pc['log2_OR'], cell_pc[col].abs())
    r_p, p_p = pearsonr(cell_pc['log2_OR'],  cell_pc[col].abs())
    dose_rows.append({'PC': pc, 'Spearman_rho': r_s, 'Spearman_p': p_s,
                      'Pearson_r': r_p, 'Pearson_p': p_p})
    star = '**' if p_s < 0.01 else ('*' if p_s < 0.05 else '')
    print(f'  {pc}: Spearman ρ={r_s:+.3f} p={p_s:.3g}{star}  |  Pearson r={r_p:+.3f} p={p_p:.3g}')

dose_df = pd.DataFrame(dose_rows)

# BH-FDR correction across all 6 PCs tested (n=11 cells; uncorrected p=0.021 for PC12)
from statsmodels.stats.multitest import multipletests
_, dose_df['q_bh'], _, _ = multipletests(dose_df['Spearman_p'], method='fdr_bh')
print('\nBH-FDR corrected (6 PCs tested, n=11 cells):')
for _, row in dose_df.iterrows():
    sig = ' [sig]' if row['q_bh'] < 0.05 else ''
    print(f"  {row['PC']}: ρ={row['Spearman_rho']:+.3f}  p={row['Spearman_p']:.3g}  q_BH={row['q_bh']:.3f}{sig}")
pc12_q = dose_df.loc[dose_df['PC'] == 'PC12', 'q_bh'].values[0]
verdict = 'significant' if pc12_q < 0.05 else 'DOES NOT survive BH correction — exploratory only'
print(f'\nPC12: uncorrected p=0.021 → BH-adjusted q={pc12_q:.3f} ({verdict})')

# Figure: OR vs PC1 and PC7 displacement scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, pc in zip(axes, ['PC1', 'PC7']):
    col = f'{pc}_displacement'
    r_s = dose_df.loc[dose_df['PC'] == pc, 'Spearman_rho'].values[0]
    p_s = dose_df.loc[dose_df['PC'] == pc, 'Spearman_p'].values[0]

    sc = ax.scatter(cell_pc['log2_OR'], cell_pc[col].abs(),
                    s=cell_pc['n_total'] * 0.3 + 50,
                    c=cell_pc['log2_OR'], cmap='YlOrRd',
                    edgecolors='black', linewidths=0.8, zorder=3)

    # Label cells by rough geographic name
    geo_labels = {
        (-25.0, -70.0): 'Atacama\n(Chile)',
        (40.0,  -80.0): 'Appalachian',
        (40.0,  -90.0): 'Midwest US',
        (30.0,   85.0): 'Himalaya',
        (25.0,  105.0): 'Yunnan',
        (25.0,  115.0): 'Guangdong',
        (30.0,  120.0): 'Yangtze',
        (25.0,  -85.0): 'Gulf',
        (50.0,   10.0): 'Central EU',
        (30.0, -120.0): 'Calif coast',
        (35.0, -125.0): 'Pacific NW',
    }
    for _, row in cell_pc.iterrows():
        lbl = geo_labels.get((row['lat_bin'], row['lon_bin']), '')
        if lbl:
            ax.annotate(lbl, (row['log2_OR'], abs(row[col])),
                        fontsize=7, ha='center', va='bottom',
                        xytext=(0, 5), textcoords='offset points')

    # Trend line
    m, b = np.polyfit(cell_pc['log2_OR'], cell_pc[col].abs(), 1)
    xs = np.linspace(cell_pc['log2_OR'].min(), cell_pc['log2_OR'].max(), 100)
    ax.plot(xs, m * xs + b, 'k--', linewidth=1.5, alpha=0.7)

    star = '**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns')
    ax.set_xlabel('log₂(Odds Ratio) — metal resistance enrichment', fontsize=11)
    ax.set_ylabel(f'|{pc} displacement from non-hotspot mean|', fontsize=11)
    ax.set_title(f'{pc}: Dose-Response\nSpearman ρ={r_s:+.3f}  {star}',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.colorbar(sc, ax=ax, label='log₂(OR)')

plt.tight_layout()
out = fig_dir / 'nb11c_ext3_dose_response.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'\nSaved: {out}')
plt.close()

Per-cell displacement vs OR:
 lat_bin  lon_bin    OR  n_total  PC1_displacement  PC7_displacement
   -25.0    -70.0 9.832      101            -0.697            -1.470
    25.0    -85.0 2.711      320             0.840            -0.068
    25.0    105.0 5.894       48            -1.684             0.223
    25.0    115.0 4.429      124            -1.617             1.144
    30.0   -120.0 2.570      247            -0.903             0.927
    30.0     85.0 6.269       39            -0.619            -2.248
    30.0    120.0 3.773      143            -3.025             0.461
    35.0   -125.0 2.480      693            -0.516             3.551
    40.0    -90.0 6.324      199            -2.198             2.146
    40.0    -80.0 7.861      132            -1.640             1.779
    50.0     10.0 2.648      298            -2.008             0.106

Correlation: |PC displacement| vs log2(OR) across 11 hotspot cells:
  PC1: Spearman ρ=+0.127 p=0.709  |  Pearson r=+0.039 p=0.909
  PC5: Spear


Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_ext3_dose_response.png


---
## Extension 4: What is PC12?

PC12 shows a significant dose-response with hotspot OR (ρ=+0.68, p=0.021) despite being only the 12th principal component. Examining the loadings and the environments that sit at PC12 extremes reveals what biological gradient it captures.

In [14]:
# Extension 4: PC12 interpretability — loadings and environment associations

# --- Loadings ---
loadings = pd.DataFrame(pca.components_.T, index=EMBD_COLS,
                         columns=[f'PC{i+1}' for i in range(20)])

print('=== PC12 top 10 loadings (|loading|) ===')
pc12_top = loadings['PC12'].abs().sort_values(ascending=False).head(10)
print(pc12_top.to_string())
print(f'\nPC12 dominated by {pc12_top.index[0]} (loading={loadings.loc[pc12_top.index[0],"PC12"]:.3f})'
      f' — next is {pc12_top.index[1]} ({loadings.loc[pc12_top.index[1],"PC12"]:.3f})')

print('\n=== PC1 top 5 loadings (for comparison) ===')
print(loadings['PC1'].abs().sort_values(ascending=False).head(5).to_string())

# --- Environment extremes ---
q10 = ae_env['PC12'].quantile(0.10)
q90 = ae_env['PC12'].quantile(0.90)
hi12 = ae_env[ae_env['PC12'] >= q90].copy()
lo12 = ae_env[ae_env['PC12'] <= q10].copy()
print(f'\nPC12 top 10% (≥{q90:.2f}): {len(hi12)} genomes')
print(f'PC12 bot 10% (≤{q10:.2f}): {len(lo12)} genomes')

# Aggregate isolation_source into coarse categories
def categorise_source(s):
    s = str(s or '').lower()
    if 'permafrost' in s:            return 'Permafrost'
    if 'aspo' in s or 'äspö' in s:  return 'Deep subsurface (Äspö HRL)'
    if 'mine' in s or 'acid mine' in s or 'amd' in s: return 'Acid mine drainage'
    if 'anaerobic digester' in s or 'activated sludge' in s or 'wastewater' in s or 'digester' in s:
        return 'Wastewater/Digester'
    if 'hydrothermal' in s or 'vent' in s: return 'Hydrothermal vent'
    if 'groundwater' in s:           return 'Groundwater'
    if 'soda lake' in s or 'hypersaline' in s: return 'Hypersaline soda lake'
    if 'microcystis' in s or 'freshwater lake' in s: return 'Freshwater lake (Microcystis)'
    if any(x in s for x in ['pork','poultry','chicken','meat','milk','cheese','food','ice-cream','yogurt']): return 'Food-associated'
    if 'hot spring' in s or 'hot spring' in s: return 'Hot spring'
    if 'root nodule' in s or 'rhizobium' in s: return 'Root nodule'
    if 'rhizosphere' in s:           return 'Rhizosphere'
    if 'soil' in s:                  return 'Soil'
    if 'sediment' in s:              return 'Sediment'
    if 'ocean' in s or 'marine' in s or 'sea' in s: return 'Marine'
    return 'Other'

hi12['env_cat'] = hi12['isolation_source'].apply(categorise_source)
lo12['env_cat'] = lo12['isolation_source'].apply(categorise_source)

hi_cat = hi12['env_cat'].value_counts(normalize=True).mul(100).round(1)
lo_cat = lo12['env_cat'].value_counts(normalize=True).mul(100).round(1)

summary = pd.DataFrame({'High PC12 (%)': hi_cat, 'Low PC12 (%)': lo_cat}).fillna(0)
summary = summary.sort_values('High PC12 (%)', ascending=False)
print('\nEnvironment category breakdown at PC12 extremes (%):')
print(summary.to_string())

# --- Figure ---
top_cats = summary.head(10).index.tolist()
sub = summary.loc[top_cats]

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(top_cats))
w = 0.38
ax.bar(x - w/2, sub['High PC12 (%)'], w, label='High PC12 (top 10%)', color='#e63946', alpha=0.85)
ax.bar(x + w/2, sub['Low PC12 (%)'],  w, label='Low PC12 (bot 10%)',  color='#457b9d', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_cats, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('% of genomes in extreme', fontsize=12)
ax.set_title('PC12 Identity: Environment Distribution at Extremes\n'
             '(PC12 = anoxic/cold/extreme-subsurface vs aerobic/saline gradient;\n'
             'permafrost + Äspö HRL dominate high end via temperature/depth, not metal cycling)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
out = fig_dir / 'nb11c_ext4_pc12_identity.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'\nSaved: {out}')
plt.close()

print('\n=== PC12 interpretation (post-hoc; exploratory) ===')
print('High PC12: permafrost active layer (29.2%), deep granite groundwater (11.6%),')
print('           wastewater/digesters (11.4%), hydrothermal vents (2.8%), AMD (2.1%)')
print('Low  PC12: hypersaline soda lake (24.7%), freshwater Microcystis (7.4%), food (12.8%)')
print()
print('→ PC12 axis: anoxic / cold / extreme-subsurface environments (high)')
print('           vs aerobic / alkaline-saline / mesic environments (low)')
print()
print('  CAVEAT: Permafrost (29.2%) + Äspö HRL (11.6%) = 40.8% of high-PC12 signal.')
print('  These environments are primarily defined by TEMPERATURE and DEPTH, not metal')
print('  cycling. True metal-associated envs (AMD + digesters) = only 13.5% of high-PC12.')
print('  PC12 encodes a cold/anoxic/isolated-subsurface niche; the metal co-occurrence')
print('  is likely because these environments also tend toward metal enrichment.')
print()
print(f'PC12 dominated by embedding dimension A10 (loading={loadings.loc["A10","PC12"]:.3f})')
print('A10 likely encodes cold/anoxic/extreme-subsurface physicochemistry.')

=== PC12 top 10 loadings (|loading|) ===
A10    0.421544
A15    0.239254
A18    0.225010
A29    0.224573
A46    0.214903
A06    0.213973
A33    0.207851
A22    0.205150
A37    0.196552
A51    0.195225

PC12 dominated by A10 (loading=0.422) — next is A15 (-0.239)

=== PC1 top 5 loadings (for comparison) ===
A31    0.248011
A25    0.235095
A17    0.229256
A02    0.226639
A36    0.219798

PC12 top 10% (≥1.75): 3748 genomes
PC12 bot 10% (≤-1.65): 3728 genomes

Environment category breakdown at PC12 extremes (%):
                               High PC12 (%)  Low PC12 (%)
env_cat                                                   
Permafrost                              29.2           0.0
Other                                   27.6          42.1
Deep subsurface (Äspö HRL)              11.6           0.0
Wastewater/Digester                     11.4           1.9
Soil                                     3.7           3.8
Hydrothermal vent                        2.8           0.4
Food-associate


Saved: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/figures/nb11c_ext4_pc12_identity.png

=== PC12 interpretation (post-hoc; exploratory) ===
High PC12: permafrost active layer (29.2%), deep granite groundwater (11.6%),
           wastewater/digesters (11.4%), hydrothermal vents (2.8%), AMD (2.1%)
Low  PC12: hypersaline soda lake (24.7%), freshwater Microcystis (7.4%), food (12.8%)

→ PC12 axis: anoxic / cold / extreme-subsurface environments (high)
           vs aerobic / alkaline-saline / mesic environments (low)

  CAVEAT: Permafrost (29.2%) + Äspö HRL (11.6%) = 40.8% of high-PC12 signal.
  These environments are primarily defined by TEMPERATURE and DEPTH, not metal
  cycling. True metal-associated envs (AMD + digesters) = only 13.5% of high-PC12.
  PC12 encodes a cold/anoxic/isolated-subsurface niche; the metal co-occurrence
  is likely because these environments also tend toward metal enrichment.

PC12 dominated by embedding dimension A10 (loadi